# IBM Db2 Vector 存储和 Vector 搜索

Db2 11.5.8 及更高版本支持向量存储和向量搜索。

向量存储允许您将向量数据存储在 Db2 表中。向量搜索允许您使用向量相似性搜索来查询这些向量。

向量存储和向量搜索功能由 Db2 的新数据类型和函数提供支持。

向量数据类型是一个新的用户定义数据类型（UDT），用于存储向量。向量函数是一组新的内置函数，用于对向量执行操作。

向量存储和向量搜索功能可用于各种用例，包括：

* **推荐系统：** 使用向量搜索来查找与用户过去行为相似的项目。
* **图像搜索：** 使用向量搜索来查找与查询图像相似的图像。
* **自然语言处理：** 使用向量搜索来查找与查询文本相似的文本。
* **异常检测：** 使用向量搜索来查找与正常数据点不同的数据点。

以下是使用 Db2 进行向量存储和向量搜索的一些示例：

```sql
-- 创建一个表来存储向量
CREATE TABLE vectors (
  id INT PRIMARY KEY,
  vector DB2GSE.VECTOR
);

-- 将向量插入表
INSERT INTO vectors (id, vector)
VALUES (1, DB2GSE.VECTOR(1.0, 2.0, 3.0));

-- 查询与向量相似的向量
SELECT id
FROM vectors
WHERE DB2GSE.COSINE_SIMILARITY(vector, DB2GSE.VECTOR(1.0, 2.0, 3.0)) > 0.9;
```

向量存储和向量搜索功能是 Db2 的一项强大功能，可用于各种用例。

LangChain 的 Db2 集成 (langchain-db2) 为使用 IBM 关系数据库 Db2 v12.1.2 及以上版本提供了向量存储和向量搜索功能，该集成遵循 MIT 许可协议分发。用户可以按原样使用提供的实现，也可以根据特定需求进行自定义。

主要功能包括：

*  带元数据的向量存储
*  向量相似性搜索和最大边际相关性搜索，支持元数据过滤选项
*  支持点积、余弦和欧几里得距离度量
*  通过索引创建和近似最近邻搜索进行性能优化。（即将添加）

## 设置

### 使用 Langchain 与 Db2 Vector Store 和 Search 的先决条件

安装 `langchain-db2` 包，这是 db2 LangChain Vector Store 和 Search 的集成包。

安装该包时，也应安装其依赖项，如 `langchain-core` 和 `ibm_db`。

In [ ]:
# pip install -U langchain-db2

### 连接到 Db2 向量存储

以下示例代码将展示如何连接到 Db2 数据库。除了上述依赖项之外，您还需要一个正在运行的 Db2 数据库实例（版本 v12.1.2+，支持向量数据类型）。

In [ ]:
import ibm_db
import ibm_db_dbi

database = ""
username = ""
password = ""

try:
    connection = ibm_db_dbi.connect(database, username, password)
    print("Connection successful!")
except Exception as e:
    print("Connection failed!")

### 导入所需依赖项

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_db2 import db2vs
from langchain_db2.db2vs import DB2VS

## 初始化

### 创建文档

This guide will walk you through the process of creating documents in your project.

本指南将引导您完成在项目中创建文档的过程。

**1. Accessing the Document Creation Interface**

1.  **访问文档创建界面**

    To begin, navigate to the "Documents" section in your project's dashboard.

    首先，导航到项目仪表板中的“文档”部分。

    *   Click on the "Documents" tab in the main navigation menu.
        *   点击主导航菜单中的“文档”选项卡。
    *   You will be presented with a list of existing documents and a prominent "Create New Document" button.
        *   您将看到现有文档列表和一个醒目的“创建新文档”按钮。

**2. Creating a New Document**

2.  **创建新文档**

    Clicking the "Create New Document" button will open a modal or a new page where you can input the details for your new document.

    点击“创建新文档”按钮将打开一个模态框或新页面，您可以在其中输入新文档的详细信息。

    **Document Title:**
    **文档标题：**

    *   Enter a clear and concise title for your document. This will be used as the primary identifier.
        *   为您的文档输入一个清晰简洁的标题。这将用作主要标识符。

    **Content:**
    **内容：**

    *   Use the rich text editor to add your document's content. You can format text, insert images, create tables, and more.
        *   使用富文本编辑器添加文档内容。您可以格式化文本、插入图片、创建表格等。
    *   For more advanced formatting or embedding external content, you can switch to the Markdown editor.
        *   对于更高级的格式设置或嵌入外部内容，您可以切换到 Markdown 编辑器。

    **Tags (Optional):**
    **标签（可选）：**

    *   Add relevant tags to categorize your document, making it easier to find later.
        *   添加相关标签以对文档进行分类，方便以后查找。

    **Permissions (Optional):**
    **权限（可选）：**

    *   If your project has granular permission settings, you can specify who can view or edit this document.
        *   如果您的项目具有细粒度的权限设置，您可以指定谁可以查看或编辑此文档。

**3. Saving and Publishing**

3.  **保存和发布**

    Once you have finished entering the details and content:

    完成详细信息和内容输入后：

    *   **Save Draft:** Click "Save Draft" to save your progress without making the document publicly visible. You can return later to continue editing.
        *   **保存草稿：** 点击“保存草稿”以保存您的进度，而无需公开文档。您可以稍后返回继续编辑。
    *   **Publish:** Click "Publish" to make the document available to the intended audience based on the set permissions.
        *   **发布：** 点击“发布”以根据设定的权限将文档提供给目标受众。

**4. Editing Existing Documents**

4.  **编辑现有文档**

    To edit a document that has already been created:

    要编辑已创建的文档：

    *   Navigate to the "Documents" section.
        *   导航到“文档”部分。
    *   Find the document you wish to edit in the list.
        *   在列表中找到您要编辑的文档。
    *   Click on the document title or an "Edit" button associated with it.
        *   点击文档标题或与之关联的“编辑”按钮。
    *   Make your changes using the editor and then click "Update" or "Save Changes".
        *   使用编辑器进行更改，然后点击“更新”或“保存更改”。

**Example:**

**示例：**

Let's create a document titled "Project Onboarding Guide".

让我们创建一个名为“项目入职指南”的文档。

1.  Go to Documents.
    1.  转到文档。
2.  Click "Create New Document".
    2.  点击“创建新文档”。
3.  Enter "Project Onboarding Guide" as the title.
    3.  输入“项目入职指南”作为标题。
4.  In the content editor, add:
    4.  在内容编辑器中，添加：

    ```markdown
    Welcome to the project! This guide will help you get started.

    ## Getting Started
    1. Clone the repository.
    2. Install dependencies: `npm install`
    3. Run the development server: `npm start`

    ## Key Contacts
    - Project Manager: Jane Doe
    - Lead Developer: John Smith
    ```

    ```markdown
    欢迎加入项目！本指南将帮助您入门。

    ## 入门
    1. 克隆仓库。
    2. 安装依赖：`npm install`
    3. 运行开发服务器：`npm start`

    ## 主要联系人
    - 项目经理：Jane Doe
    - 主管开发人员：John Smith
    ```
5.  Click "Publish".
    5.  点击“发布”。

Your "Project Onboarding Guide" document is now created and accessible.

您的“项目入职指南”文档现已创建并可访问。

In [ ]:
# Define a list of documents
documents_json_list = [
    {
        "id": "doc_1_2_P4",
        "text": "Db2 handles LOB data differently than other kinds of data. As a result, you sometimes need to take additional actions when you define LOB columns and insert the LOB data.",
        "link": "https://www.ibm.com/docs/en/db2-for-zos/12?topic=programs-storing-lob-data-in-tables",
    },
    {
        "id": "doc_11.1.0_P1",
        "text": "Db2® column-organized tables add columnar capabilities to Db2 databases, which include data that is stored with column organization and vector processing of column data. Using this table format with star schema data marts provides significant improvements to storage, query performance, and ease of use through simplified design and tuning.",
        "link": "https://www.ibm.com/docs/en/db2/11.1.0?topic=organization-column-organized-tables",
    },
    {
        "id": "id_22.3.4.3.1_P2",
        "text": "Data structures are elements that are required to use Db2®. You can access and use these elements to organize your data. Examples of data structures include tables, table spaces, indexes, index spaces, keys, views, and databases.",
        "link": "https://www.ibm.com/docs/en/zos-basic-skills?topic=concepts-db2-data-structures",
    },
    {
        "id": "id_3.4.3.1_P3",
        "text": "Db2® maintains a set of tables that contain information about the data that Db2 controls. These tables are collectively known as the catalog. The catalog tables contain information about Db2 objects such as tables, views, and indexes. When you create, alter, or drop an object, Db2 inserts, updates, or deletes rows of the catalog that describe the object.",
        "link": "https://www.ibm.com/docs/en/zos-basic-skills?topic=objects-db2-catalog",
    },
]

In [ ]:
# Create Langchain Documents

documents_langchain = []

for doc in documents_json_list:
    metadata = {"id": doc["id"], "link": doc["link"]}
    doc_langchain = Document(page_content=doc["text"], metadata=metadata)
    documents_langchain.append(doc_langchain)

### 创建具有不同距离度量的向量存储

首先，我们将创建三个具有不同距离策略的向量存储。

（您可以手动连接到 Db2 数据库，并将看到三个表：
Documents_DOT、Documents_COSINE 和 Documents_EUCLIDEAN。）

In [ ]:
# Create Db2 Vector Stores using different distance strategies

# When using our API calls, start by initializing your vector store with a subset of your documents
# through from_documents(), then incrementally add more documents using add_texts().
# This approach prevents system overload and ensures efficient document processing.

model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store_dot = DB2VS.from_documents(
    documents_langchain,
    model,
    client=connection,
    table_name="Documents_DOT",
    distance_strategy=DistanceStrategy.DOT_PRODUCT,
)
vector_store_max = DB2VS.from_documents(
    documents_langchain,
    model,
    client=connection,
    table_name="Documents_COSINE",
    distance_strategy=DistanceStrategy.COSINE,
)
vector_store_euclidean = DB2VS.from_documents(
    documents_langchain,
    model,
    client=connection,
    table_name="Documents_EUCLIDEAN",
    distance_strategy=DistanceStrategy.EUCLIDEAN_DISTANCE,
)

## 管理向量存储

This guide explains how to manage your vector stores.

本指南将介绍如何管理您的向量存储。

Vector stores are databases that store vector embeddings. They are used to power semantic search and other AI-powered features.

向量存储是存储向量嵌入的数据库。它们用于支持语义搜索和其他 AI 驱动的功能。

### Creating a vector store

### 创建向量存储

To create a vector store, you need to provide a name for the store and specify the embedding model to use.

要创建向量存储，您需要提供存储的名称并指定要使用的嵌入模型。

```python
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# set up ChromaDB client
chroma_client = chromadb.Client()
chroma_collection = chroma_client.create_collection("quickstart")

# set up ChromaVectorStore
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# load documents
documents = [TextNode(text="This is a sample document.")]

# set up storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# build index
index = VectorStoreIndex(nodes=documents, storage_context=storage_context)
```

### Loading a vector store

### 加载向量存储

To load a vector store, you need to provide the path to the storage directory.

要加载向量存储，您需要提供存储目录的路径。

```python
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# set up ChromaDB client
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_collection("quickstart")

# set up ChromaVectorStore
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# set up storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# load index
index = load_index_from_storage(storage_context=storage_context)
```

### Adding data to a vector store

### 向向量存储添加数据

To add data to a vector store, you can use the `insert` method of the `VectorStore` object.

要向向量存储添加数据，您可以使用 `VectorStore` 对象的 `insert` 方法。

```python
# add more documents
new_documents = [TextNode(text="This is another sample document.")]
index.insert(nodes=new_documents)
```

### Querying a vector store

### 查询向量存储

To query a vector store, you can use the `query` method of the `VectorStoreIndex` object.

要查询向量存储，您可以使用 `VectorStoreIndex` 对象的 `query` 方法。

```python
query_engine = index.as_query_engine()
response = query_engine.query("What is this document about?")
print(response)
```

### Deleting data from a vector store

### 从向量存储中删除数据

To delete data from a vector store, you can use the `delete` method of the `VectorStore` object.

要从向量存储中删除数据，您可以使用 `VectorStore` 对象的 `delete` 方法。

```python
# delete a document
index.delete(nodes=[documents[0]])
```

### Updating data in a vector store

### 更新向量存储中的数据

To update data in a vector store, you can use the `update` method of the `VectorStore` object.

要更新向量存储中的数据，您可以使用 `VectorStore` 对象的 `update` 方法。

```python
# update a document
documents[0].text = "This is an updated sample document."
index.update(nodes=[documents[0]])

### 演示文本的添加和删除操作，以及基本的相似性搜索

This example demonstrates how to add and delete texts from a vector store, and how to perform basic similarity searches.

本示例演示了如何从向量存储中添加和删除文本，以及如何执行基本的相似性搜索。

```python
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import OpenAIEmbeddings

# Initialize the embedding model
# 初始化嵌入模型
embeddings = OpenAIEmbeddings()

# Create a Chroma vector store
# 创建一个 Chroma 向量存储
# You can specify a path to persist the database
# 您可以指定一个路径来持久化数据库
# For this example, we'll use an in-memory store
# 在本示例中，我们将使用内存中的存储
vectorstore = Chroma(embedding_function=embeddings)

# Add documents to the vector store
# 将文档添加到向量存储
texts = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]
metadatas = [
    {"source": "doc1"},
    {"source": "doc2"},
    {"source": "doc3"},
    {"source": "doc4"},
]
doc_ids = ["doc1", "doc2", "doc3", "doc4"]

# Add documents with their corresponding metadata and IDs
# 添加带有相应元数据和 ID 的文档
vectorstore.add_documents(texts, metadatas=metadatas, ids=doc_ids)

# Perform a similarity search
# 执行相似性搜索
query = "What is this document?"
results = vectorstore.similarity_search(query)
print("Similarity search results:")
# 打印相似性搜索结果
for doc in results:
    print(f"- {doc.page_content} (Metadata: {doc.metadata})")

# Delete a document by its ID
# 通过 ID 删除文档
print("\nDeleting document with ID 'doc2'...")
# 打印正在删除 ID 为 'doc2' 的文档
vectorstore.delete(ids=["doc2"])
print("Document deleted.")
# 打印文档已删除

# Perform another similarity search to see the effect of deletion
# 再执行一次相似性搜索，查看删除的效果
print("\nPerforming similarity search after deletion:")
# 打印删除后执行相似性搜索
results_after_delete = vectorstore.similarity_search(query)
print("Similarity search results after deletion:")
# 打印删除后相似性搜索结果
for doc in results_after_delete:
    print(f"- {doc.page_content} (Metadata: {doc.metadata})")

# Add a new document
# 添加一个新文档
print("\nAdding a new document...")
# 打印正在添加一个新文档
new_text = "This is a new document added later."
new_metadata = {"source": "doc5"}
new_id = "doc5"
vectorstore.add_documents([new_text], metadatas=[new_metadata], ids=[new_id])
print("New document added.")
# 打印新文档已添加

# Perform a final similarity search
# 执行最后的相似性搜索
print("\nPerforming similarity search after adding a new document:")
# 打印添加新文档后执行相似性搜索
results_after_add = vectorstore.similarity_search(query)
print("Similarity search results after adding a new document:")
# 打印添加新文档后相似性搜索结果
for doc in results_after_add:
    print(f"- {doc.page_content} (Metadata: {doc.metadata})")

In [ ]:
def manage_texts(vector_stores):
    """
    Adds texts to each vector store, demonstrates error handling for duplicate additions,
    and performs deletion of texts. Showcases similarity searches and index creation for each vector store.

    Args:
    - vector_stores (list): A list of DB2VS instances.
    """
    texts = ["Rohan", "Shailendra"]
    metadata = [
        {"id": "100", "link": "Document Example Test 1"},
        {"id": "101", "link": "Document Example Test 2"},
    ]

    for i, vs in enumerate(vector_stores, start=1):
        # Adding texts
        try:
            vs.add_texts(texts, metadata)
            print(f"\n\n\nAdd texts complete for vector store {i}\n\n\n")
        except Exception as ex:
            print(f"\n\n\nExpected error on duplicate add for vector store {i}\n\n\n")

        # Deleting texts using the value of 'id'
        vs.delete([metadata[0]["id"], metadata[1]["id"]])
        print(f"\n\n\nDelete texts complete for vector store {i}\n\n\n")

        # Similarity search
        results = vs.similarity_search("How are LOBS stored in Db2 Database", 2)
        print(f"\n\n\nSimilarity search results for vector store {i}: {results}\n\n\n")


vector_store_list = [
    vector_store_dot,
    vector_store_max,
    vector_store_euclidean,
]
manage_texts(vector_store_list)

## 查询向量存储

### 展示向量存储的高级搜索，带或不带属性过滤

通过过滤，我们仅选择文档 ID 101，不选择其他任何内容

In [ ]:
# Conduct advanced searches
def conduct_advanced_searches(vector_stores):
    query = "How are LOBS stored in Db2 Database"
    # Constructing a filter for direct comparison against document metadata
    # This filter aims to include documents whose metadata 'id' is exactly '101'
    filter_criteria = {"id": ["101"]}  # Direct comparison filter

    for i, vs in enumerate(vector_stores, start=1):
        print(f"\n--- Vector Store {i} Advanced Searches ---")
        # Similarity search without a filter
        print("\nSimilarity search results without filter:")
        print(vs.similarity_search(query, 2))

        # Similarity search with a filter
        print("\nSimilarity search results with filter:")
        print(vs.similarity_search(query, 2, filter=filter_criteria))

        # Similarity search with relevance score
        print("\nSimilarity search with relevance score:")
        print(vs.similarity_search_with_score(query, 2))

        # Similarity search with relevance score with filter
        print("\nSimilarity search with relevance score with filter:")
        print(vs.similarity_search_with_score(query, 2, filter=filter_criteria))

        # Max marginal relevance search
        print("\nMax marginal relevance search results:")
        print(vs.max_marginal_relevance_search(query, 2, fetch_k=20, lambda_mult=0.5))

        # Max marginal relevance search with filter
        print("\nMax marginal relevance search results with filter:")
        print(
            vs.max_marginal_relevance_search(
                query, 2, fetch_k=20, lambda_mult=0.5, filter=filter_criteria
            )
        )


conduct_advanced_searches(vector_store_list)

## 用于检索增强生成的使用方法

Retrieval-augmented generation (RAG) is a technique that combines the power of large language models (LLMs) with external knowledge bases. This allows LLMs to access and incorporate up-to-date, domain-specific information into their responses, leading to more accurate, relevant, and informative outputs.

检索增强生成（RAG）是一种将大型语言模型（LLM）的能力与外部知识库相结合的技术。这使得 LLM 能够访问并整合最新的、特定领域的知识到它们的响应中，从而产生更准确、更相关、信息更丰富的输出。

### Key Components of RAG

RAG systems typically consist of the following key components:

### RAG 的关键组成部分

RAG 系统通常包含以下关键组成部分：

1.  **Retriever:** This component is responsible for searching and retrieving relevant information from an external knowledge base (e.g., a database, a collection of documents, or the internet). The retriever uses various techniques, such as keyword matching, semantic search, or vector embeddings, to find the most pertinent information related to the user's query.

    1.  **检索器 (Retriever):** 此组件负责从外部知识库（例如，数据库、文档集合或互联网）中搜索和检索相关信息。检索器使用各种技术，如关键词匹配、语义搜索或向量嵌入，来查找与用户查询最相关的信息。

2.  **Generator:** This component is typically a large language model (LLM) that takes the user's query and the retrieved information as input. The LLM then synthesizes this information to generate a coherent and contextually relevant response.

    2.  **生成器 (Generator):** 此组件通常是一个大型语言模型（LLM），它接收用户查询和检索到的信息作为输入。然后，LLM 会综合这些信息，生成连贯且与上下文相关的响应。

### How RAG Works

The RAG process generally follows these steps:

### RAG 的工作原理

RAG 过程通常遵循以下步骤：

1.  **User Query:** The user submits a query or question to the RAG system.

    1.  **用户查询 (User Query):** 用户向 RAG 系统提交查询或问题。

2.  **Information Retrieval:** The retriever component searches the external knowledge base for information relevant to the user's query.

    2.  **信息检索 (Information Retrieval):** 检索器组件在外部知识库中搜索与用户查询相关的信息。

3.  **Context Augmentation:** The retrieved information is then used to augment the original user query, creating a more comprehensive prompt for the LLM.

    3.  **上下文增强 (Context Augmentation):** 检索到的信息随后用于增强原始用户查询，为 LLM 创建一个更全面的提示。

4.  **Response Generation:** The generator (LLM) receives the augmented prompt and generates a response based on both its internal knowledge and the provided external information.

    4.  **响应生成 (Response Generation):** 生成器（LLM）接收增强后的提示，并根据其内部知识和提供的外部信息生成响应。

### Benefits of Using RAG

*   **Improved Accuracy:** By grounding responses in factual external data, RAG reduces the likelihood of generating inaccurate or fabricated information (hallucinations).
*   **Up-to-date Information:** RAG allows LLMs to access current information, overcoming the limitation of static training data.
*   **Domain-Specific Knowledge:** It enables LLMs to leverage specialized knowledge from specific domains, leading to more expert-level responses.
*   **Transparency and Explainability:** In some implementations, RAG can provide citations or sources for the information used, enhancing transparency.

### 使用 RAG 的优势

*   **提高准确性:** 通过将响应基于事实性的外部数据，RAG 降低了生成不准确或虚假信息（幻觉）的可能性。
*   **最新信息:** RAG 允许 LLM 访问当前信息，克服了静态训练数据的局限性。
*   **领域特定知识:** 它使 LLM 能够利用特定领域的专业知识，从而产生更具专家水平的响应。
*   **透明度和可解释性:** 在某些实现中，RAG 可以为所使用的信息提供引用或来源，从而提高透明度。

### Use Cases for RAG

RAG can be applied in a wide range of applications, including:

### RAG 的用例

RAG 可应用于广泛的场景，包括：

*   **Question Answering:** Providing accurate and detailed answers to user questions by retrieving information from knowledge bases.
*   **Content Creation:** Generating articles, summaries, or reports that are informed by specific data sources.
*   **Chatbots and Virtual Assistants:** Enhancing conversational AI with real-time information and domain expertise.
*   **Research and Analysis:** Assisting researchers by summarizing and synthesizing information from large volumes of text.
*   **Customer Support:** Providing customers with accurate information and solutions based on product documentation or FAQs.

*   **问答 (Question Answering):** 通过从知识库检索信息，为用户问题提供准确而详细的答案。
*   **内容创作 (Content Creation):** 生成基于特定数据源的信息文章、摘要或报告。
*   **聊天机器人和虚拟助手 (Chatbots and Virtual Assistants):** 通过实时信息和领域专业知识增强会话式 AI。
*   **研究和分析 (Research and Analysis):** 通过总结和综合大量文本信息来协助研究人员。
*   **客户支持 (Customer Support):** 根据产品文档或常见问题解答，为客户提供准确的信息和解决方案。

### Implementing RAG

Implementing a RAG system typically involves:

### 实现 RAG

实现 RAG 系统通常涉及：

1.  **Choosing a Knowledge Base:** Selecting and preparing the external data source (e.g., a vector database, a document store).
2.  **Setting up a Retriever:** Configuring a retrieval mechanism (e.g., using embedding models like Sentence-BERT or OpenAI embeddings).
3.  **Integrating with an LLM:** Connecting the retriever to a generative LLM (e.g., GPT-3.5, GPT-4, or open-source alternatives).
4.  **Prompt Engineering:** Crafting effective prompts that guide the LLM to utilize the retrieved context appropriately.

1.  **选择知识库 (Choosing a Knowledge Base):** 选择和准备外部数据源（例如，向量数据库、文档存储）。
2.  **设置检索器 (Setting up a Retriever):** 配置检索机制（例如，使用 Sentence-BERT 或 OpenAI embeddings 等嵌入模型）。
3.  **与 LLM 集成 (Integrating with an LLM):** 将检索器连接到生成式 LLM（例如，GPT-3.5、GPT-4 或开源替代品）。
4.  **提示工程 (Prompt Engineering):** 制作有效的提示，指导 LLM 正确利用检索到的上下文。

By leveraging RAG, developers can build more intelligent and reliable AI applications that go beyond the inherent knowledge of LLMs.

通过利用 RAG，开发人员可以构建更智能、更可靠的 AI 应用程序，这些应用程序能够超越 LLM 的固有知识。

## API 参考